# PullRequestEvent pipeline test (`02_pull_request_events`)

This notebook tests `02_pull_request_events` with a small synthetic Bronze table in the schema `github_lakehouse.test_pull_request`. It runs the real notebook with `%run` and `test_schema = test_pull_request`, so **production tables are never written**. The last check section proves that by comparing the production table versions before and after.

**Before you run it**, the production tables must exist, because the test tables are created with `CREATE TABLE ... LIKE`:

1. `11_silver_audit_table` (audit table; it already exists from the PushEvent work)
2. cells 1–6 of `02_pull_request_events` (they create `silver.pull_request_events` and `quarantine.quarantine_pull_request_event`)

Every check prints `PASS` or `FAIL`. A section stops with an error if any of its checks failed. The last cell drops the test schema.

The shared pipeline (parse errors, core IDs, duplicates, all-dates mode, partial failure) is already covered by `tests/test_08_fork_events`. This test covers what is specific to PullRequestEvent: the eleven PR fields are read and converted (`pr_number` from `payload.number`), several events of one PR are all kept, `pr_closed_at` and `pr_merged_at` are natural NULLs that warn only when invalid or required, an impossible close or merge time is warned about and set to `NULL`, a negative size count is warned about and set to `NULL`, an unknown `pr_merged` stays `NULL` (never `false`), an unexpected action is kept with a warning, and the quarantine table keeps the PR fields and their original text.

## B1. Setup: test schema, empty tables and helpers

In [0]:
import json
from datetime import date, datetime, timezone

TEST_SCHEMA = "test_pull_request"
assert TEST_SCHEMA.startswith("test_")   # never drop anything outside a test schema

def T(name):
    return f"github_lakehouse.{TEST_SCHEMA}.{name}"

TEST_BRONZE = T("github_events_raw")
TEST_SILVER = T("pull_request_events")
TEST_QUARANTINE = T("quarantine_pull_request_event")
TEST_AUDIT = T("silver_pipeline_runs")

PROD_TABLES = [
    "github_lakehouse.bronze.github_events_raw",
    "github_lakehouse.silver.pull_request_events",
    "github_lakehouse.quarantine.quarantine_pull_request_event",
    "github_lakehouse.audit.silver_pipeline_runs",
]

# The test tables copy the production definitions, so those must exist first.
missing = [t for t in PROD_TABLES if not spark.catalog.tableExists(t)]
if missing:
    raise RuntimeError(
        f"Missing production tables: {missing}. Run 11_silver_audit_table "
        "and cells 1-6 of 02_pull_request_events (table creation) first."
    )

spark.sql(f"CREATE SCHEMA IF NOT EXISTS github_lakehouse.{TEST_SCHEMA}")
for t in [TEST_BRONZE, TEST_SILVER, TEST_QUARANTINE, TEST_AUDIT]:
    spark.sql(f"DROP TABLE IF EXISTS {t}")

# Bronze: only the four columns the pipeline reads.
spark.sql(f"""
    CREATE TABLE {TEST_BRONZE} (
        raw_json STRING, _source_file STRING,
        _source_date DATE, _ingested_at TIMESTAMP
    ) USING DELTA
""")
# Targets: exact copies of the production table definitions (no data).
spark.sql(f"CREATE TABLE {TEST_SILVER} LIKE github_lakehouse.silver.pull_request_events")
spark.sql(f"CREATE TABLE {TEST_QUARANTINE} LIKE github_lakehouse.quarantine.quarantine_pull_request_event")
spark.sql(f"CREATE TABLE {TEST_AUDIT} LIKE github_lakehouse.audit.silver_pipeline_runs")


# ---- assertion helpers ----
_results = []

def check(label, actual, expected):
    ok = actual == expected
    _results.append((label, ok))
    print(f"{'PASS' if ok else 'FAIL'}  {label}: actual={actual!r} expected={expected!r}")

def finish(section):
    failed = [label for label, ok in _results if not ok]
    _results.clear()
    if failed:
        raise AssertionError(f"{section}: {len(failed)} check(s) failed: {failed}")
    print(f"\n{section}: ALL CHECKS PASSED")

def check_audit(expected_counts, expected_quality=None,
                expected_status="SUCCEEDED", reconciled=True):
    # `audit` is the dict created by the pipeline's audit cell via %run.
    check("audit status", audit["status"], expected_status)
    check("audit reconciliation_passed", audit["reconciliation_passed"], reconciled)
    for k, v in expected_counts.items():
        check(f"audit {k}", audit.get(k), v)
    for k, v in (expected_quality or {}).items():
        check(f"quality {k}", audit["quality_metrics"].get(k), v)

def rows(table, where="true"):
    return [r.asDict() for r in spark.table(table).where(where).collect()]

def one_silver(event_id):
    found = rows(TEST_SILVER, f"event_id = '{event_id}'")
    check(f"silver has exactly one '{event_id}'", len(found), 1)
    return found[0] if found else {}

def table_version(t):
    return spark.sql(f"DESCRIBE HISTORY {t} LIMIT 1").collect()[0]["version"]

print("Setup complete:", TEST_BRONZE, TEST_SILVER, TEST_QUARANTINE, TEST_AUDIT)

## B2. Safety snapshot of the production tables

In [0]:
prod_versions_before = {t: table_version(t) for t in PROD_TABLES}
prod_versions_before

## B3. Load the synthetic Bronze data

The default event is an `opened` PR `id 9001`, `number 42`, created `2024-01-31T08:00:00Z`, not closed, not merged, with 3 commits, 10 additions, 2 deletions and 1 changed file. `closed()` and `merged()` build a closed PR (closed `2024-02-01T09:00:00Z`) and a merged one (merged at the same time).

| Row | Case | Expected result |
|---|---|---|
| `p1` | clean `opened` PR | Silver, close and merge times NULL without a warning, `pr_merged = false` |
| `p2` | the same PR, `closed` and merged | Silver, all times set, no warning; `p1` and `p2` are both kept |
| `p3` | `closed`, `closed_at` missing | Silver, warning `MISSING: pr_closed_at` |
| `p4` | `merged = true`, `merged_at` missing | Silver, `pr_merged` stays true, warning `MISSING: pr_merged_at` |
| `p5` | `closed`, `closed_at = "not-a-date"` | Silver, warning `INVALID_CAST: pr_closed_at` |
| `p6` | merged before it was created | Silver, `pr_merged_at` NULL, `pr_merged` still true, warning `PR_MERGED_BEFORE_CREATED` |
| `p7` | closed before it was created | Silver, `pr_closed_at` NULL, warning `PR_CLOSED_BEFORE_CREATED` |
| `p8` | `additions = -5` | Silver, `pr_additions` NULL, other counts kept, warning `INVALID_NEGATIVE_PR_ADDITIONS` |
| `p9` | `merged` missing | Silver, `pr_merged` NULL (not false), warning `MISSING: pr_merged` |
| `p10` | `merged = "yes"` | Silver, `pr_merged` NULL, warning `INVALID_CAST: pr_merged` |
| `p11` | `action = synchronize` | Silver, value kept, warning `UNEXPECTED_VALUE: pr_action` |
| `p12` | `payload.number` missing | Silver, warning `MISSING: pr_number` |
| `p13` | `commits = "abc"` | Silver, warning `INVALID_CAST: pr_commits` |
| `p14` | no `pull_request` object | Silver, seven `MISSING` warnings (none for the two natural-NULL times) |
| `p15` | `pull_request` is text, not an object | quarantine `JSON_PARSE_ERROR` |
| `p16` | `id` missing | quarantine `MISSING: event_id`, PR fields and original text kept |
| `p17` (twice) | duplicate `event_id`, older and newer ingestion | one Silver row, from the newer file |
| `p18` | clean `reopened` PR | Silver, no warning |
| `x1` | a PullRequestReviewEvent | ignored (not read by this notebook) |

In [0]:
F1 = "synthetic/2024-02-01-0.json.gz"
F2 = "synthetic/2024-02-01-1.json.gz"
D1 = date(2024, 2, 1)
I1 = datetime(2024, 3, 1, tzinfo=timezone.utc)   # earlier ingestion
I2 = datetime(2024, 3, 2, tzinfo=timezone.utc)   # later ingestion

CLOSED_AT = "2024-02-01T09:00:00Z"
BEFORE_CREATED = "2024-01-30T08:00:00Z"   # earlier than the default created_at

def pull(drop=(), **changes):
    """A pull_request object. Defaults = an open, unmerged PR."""
    p = {"id": 9001, "created_at": "2024-01-31T08:00:00Z", "closed_at": None,
         "merged_at": None, "merged": False, "commits": 3, "additions": 10,
         "deletions": 2, "changed_files": 1}
    p.update(changes)
    for key in drop:
        p.pop(key)
    return p

def closed(**changes):
    return pull(**{"closed_at": CLOSED_AT, **changes})

def merged(**changes):
    return pull(**{"closed_at": CLOSED_AT, "merged_at": CLOSED_AT, "merged": True, **changes})

def ev(event_id, type_="PullRequestEvent", created_at="2024-02-01T10:00:00Z",
       actor=None, repo=None, action="opened", number=42, pr=None, payload=None):
    """One GitHub-style event as a JSON string. Defaults = a clean opened PR."""
    if payload is None:
        payload = {"action": action, "number": number,
                   "pull_request": pull() if pr is None else pr}
        if number is None:
            payload.pop("number")
    event = {
        "id": event_id,
        "type": type_,
        "created_at": created_at,
        "actor": {"id": 101, "login": "alice"} if actor is None else actor,
        "repo": {"id": 201, "name": "octo/repo"} if repo is None else repo,
        "payload": payload,
    }
    if event_id is None:
        event.pop("id")
    return json.dumps(event)

synthetic_rows = [
    # (raw_json, _source_file, _source_date, _ingested_at)                          # case
    (ev("p1"),                                                       F1, D1, I1),   # clean opened
    (ev("p2", action="closed", pr=merged()),                         F1, D1, I1),   # same PR, closed+merged
    (ev("p3", action="closed", pr=closed(drop=["closed_at"])),       F1, D1, I1),   # closed, no closed_at
    (ev("p4", action="closed", pr=merged(drop=["merged_at"])),       F1, D1, I1),   # merged, no merged_at
    (ev("p5", action="closed", pr=closed(closed_at="not-a-date")),   F1, D1, I1),   # closed_at invalid
    (ev("p6", action="closed", pr=merged(merged_at=BEFORE_CREATED)), F1, D1, I1),   # merged before created
    (ev("p7", action="closed", pr=closed(closed_at=BEFORE_CREATED)), F1, D1, I1),   # closed before created
    (ev("p8", pr=pull(additions=-5)),                                F1, D1, I1),   # negative additions
    (ev("p9", pr=pull(drop=["merged"])),                             F1, D1, I1),   # merged missing
    (ev("p10", pr=pull(merged="yes")),                               F1, D1, I1),   # merged invalid
    (ev("p11", action="synchronize"),                                F1, D1, I1),   # unexpected action
    (ev("p12", number=None),                                         F1, D1, I1),   # pr_number missing
    (ev("p13", pr=pull(commits="abc")),                              F1, D1, I1),   # commits invalid
    (ev("p14", payload={"action": "opened", "number": 42}),          F1, D1, I1),   # no pull_request
    (ev("p15", pr="oops"),                                           F1, D1, I1),   # parse error
    (ev(None),                                                       F1, D1, I1),   # event_id missing (p16)
    (ev("p17", pr=pull(id=1701)),                                    F1, D1, I1),   # duplicate, older
    (ev("p17", pr=pull(id=1702)),                                    F2, D1, I2),   # duplicate, newer -> wins
    (ev("p18", action="reopened"),                                   F1, D1, I1),   # clean reopened
    (ev("x1", type_="PullRequestReviewEvent"),                       F1, D1, I1),   # other event type
]

(spark.createDataFrame(synthetic_rows, spark.table(TEST_BRONZE).schema)
    .write.mode("overwrite").saveAsTable(TEST_BRONZE))

check("synthetic bronze rows", spark.table(TEST_BRONZE).count(), 20)
finish("B3 load")

## B4. Run 1: single date 2024-02-01

In [0]:
%run ../02_pull_request_events $test_schema="test_pull_request" $source_date="2024-02-01" $run_id="test-run1"

## B5. Run 1 checks

In [0]:
QUALITY_RUN1 = {
    "null_event_timestamp": 0, "null_actor_id": 0, "null_actor_login": 0,
    "null_repo_name": 0,
    "null_pr_action": 0, "null_pr_number": 1, "null_pr_id": 1,
    "null_pr_id_and_pr_number": 0, "null_pr_created_at": 1,
    "null_pr_closed_at": 13, "null_pr_merged_at": 15, "null_pr_merged": 3,
    "null_pr_commits": 2, "null_pr_additions": 1, "null_pr_deletions": 1,
    "null_pr_changed_files": 1,
    "closed_without_closed_at": 2, "merged_without_merged_at": 1,
    "pr_closed_before_created": 1, "pr_merged_before_created": 1,
    "negative_pr_commits": 0, "negative_pr_additions": 1,
    "negative_pr_deletions": 0, "negative_pr_changed_files": 0,
    "unexpected_pr_action": 1,
    "missing_lineage": 0,
}

# --- tables and audit identity: the run used the test tables only
check("resolved silver", SILVER_TABLE, TEST_SILVER)
check("resolved quarantine", QUARANTINE_TABLE, TEST_QUARANTINE)
check("audit pipeline_name", audit["pipeline_name"], "silver_pull_request_events")
check("audit event_type", audit["event_type"], "PullRequestEvent")

# --- audit: 19 PullRequestEvent rows (x1 is another type)
check_audit({
    "bronze_input_rows": 19, "hard_null_quarantine_rows": 2,
    "validation_quarantine_rows": 0, "validated_rows": 17,
    "duplicate_rows_removed": 1, "silver_candidate_rows": 16,
    "silver_rows_inserted": 16, "silver_rows_already_present": 0,
    "quarantine_candidate_rows": 2, "quarantine_duplicates_removed": 0,
    "quarantine_rows_inserted": 2,
}, QUALITY_RUN1)
check("quality metric names", sorted(audit["quality_metrics"]), sorted(QUALITY_RUN1))

# --- Silver schema: the contract columns, PR fields before lineage
check("silver columns", spark.table(TEST_SILVER).columns, [
    "event_id", "event_type", "event_timestamp", "event_date",
    "actor_id", "actor_login", "repo_id", "repo_name",
    "pr_action", "pr_number", "pr_id", "pr_created_at", "pr_closed_at",
    "pr_merged_at", "pr_merged", "pr_commits", "pr_additions", "pr_deletions",
    "pr_changed_files",
    "_source_file", "_source_date", "_ingested_at", "quarantine_warning",
])

# --- Silver contents
check("silver event_ids",
      sorted(r["event_id"] for r in rows(TEST_SILVER)),
      sorted(["p1", "p2", "p3", "p4", "p5", "p6", "p7", "p8", "p9", "p10",
              "p11", "p12", "p13", "p14", "p17", "p18"]))

def times(event_id):
    """The three PR timestamps as UTC text, so the check does not depend on Python time zones."""
    found = (spark.table(TEST_SILVER).where(f"event_id = '{event_id}'")
             .selectExpr(*[f"date_format({c}, 'yyyy-MM-dd HH:mm:ss') AS {c}"
                           for c in ["pr_created_at", "pr_closed_at", "pr_merged_at"]])
             .collect())
    return tuple(found[0]) if found else "row not found"

def sizes(r):
    return (r.get("pr_commits"), r.get("pr_additions"), r.get("pr_deletions"),
            r.get("pr_changed_files"))

r = one_silver("p1")
check("p1 ids and action", (r.get("pr_action"), r.get("pr_number"), r.get("pr_id")),
      ("opened", 42, 9001))
check("p1 times", times("p1"), ("2024-01-31 08:00:00", None, None))
check("p1 merged / sizes", (r.get("pr_merged"), sizes(r)), (False, (3, 10, 2, 1)))
check("p1 warning (natural NULLs are silent)", r.get("quarantine_warning"), None)

r = one_silver("p2")
check("p2 times", times("p2"), ("2024-01-31 08:00:00", "2024-02-01 09:00:00", "2024-02-01 09:00:00"))
check("p2 merged", r.get("pr_merged"), True)
check("p2 warning", r.get("quarantine_warning"), None)
check("p1 and p2 both kept for one pr_id",
      sorted(r["event_id"] for r in rows(TEST_SILVER, "pr_id = 9001 AND event_id IN ('p1', 'p2')")),
      ["p1", "p2"])

r = one_silver("p3")
check("p3 warning", r.get("quarantine_warning"), "MISSING: pr_closed_at")

r = one_silver("p4")
check("p4 still merged", r.get("pr_merged"), True)
check("p4 merged_at", times("p4")[2], None)
check("p4 warning", r.get("quarantine_warning"), "MISSING: pr_merged_at")

r = one_silver("p5")
check("p5 closed_at", times("p5")[1], None)
check("p5 warning", r.get("quarantine_warning"), "INVALID_CAST: pr_closed_at")

r = one_silver("p6")
check("p6 times: only merged_at removed", times("p6"),
      ("2024-01-31 08:00:00", "2024-02-01 09:00:00", None))
check("p6 still merged", r.get("pr_merged"), True)
check("p6 warning", r.get("quarantine_warning"), "PR_MERGED_BEFORE_CREATED")

r = one_silver("p7")
check("p7 times: only closed_at removed", times("p7"), ("2024-01-31 08:00:00", None, None))
check("p7 warning", r.get("quarantine_warning"), "PR_CLOSED_BEFORE_CREATED")

r = one_silver("p8")
check("p8 sizes: only additions removed", sizes(r), (3, None, 2, 1))
check("p8 warning", r.get("quarantine_warning"), "INVALID_NEGATIVE_PR_ADDITIONS")

r = one_silver("p9")
check("p9 merged is NULL, not False", r.get("pr_merged"), None)
check("p9 warning", r.get("quarantine_warning"), "MISSING: pr_merged")

r = one_silver("p10")
check("p10 merged", r.get("pr_merged"), None)
check("p10 warning", r.get("quarantine_warning"), "INVALID_CAST: pr_merged")

r = one_silver("p11")
check("p11 action kept unchanged", r.get("pr_action"), "synchronize")
check("p11 warning", r.get("quarantine_warning"), "UNEXPECTED_VALUE: pr_action")

r = one_silver("p12")
check("p12 pr_number NULL, pr_id kept", (r.get("pr_number"), r.get("pr_id")), (None, 9001))
check("p12 warning", r.get("quarantine_warning"), "MISSING: pr_number")

r = one_silver("p13")
check("p13 commits", r.get("pr_commits"), None)
check("p13 warning", r.get("quarantine_warning"), "INVALID_CAST: pr_commits")

r = one_silver("p14")
check("p14 fields", (r.get("pr_action"), r.get("pr_number"), r.get("pr_id"), r.get("pr_merged"), sizes(r)),
      ("opened", 42, None, None, (None, None, None, None)))
check("p14 warnings", r.get("quarantine_warning"),
      "MISSING: pr_id; MISSING: pr_created_at; MISSING: pr_merged; MISSING: pr_commits; "
      "MISSING: pr_additions; MISSING: pr_deletions; MISSING: pr_changed_files")

r = one_silver("p17")
check("p17 winner = newer ingestion", (r.get("pr_id"), r.get("_source_file")), (1702, F2))

r = one_silver("p18")
check("p18 action", r.get("pr_action"), "reopened")
check("p18 warning", r.get("quarantine_warning"), None)

# --- Quarantine contents
q = rows(TEST_QUARANTINE)
check("quarantine reasons",
      sorted(r["quarantine_reason"] for r in q), ["JSON_PARSE_ERROR", "MISSING: event_id"])
# Quarantine rows are identified by their raw_json, because parsed fields
# (including event_id) may be NULL.
parse_error = [r for r in q if r["quarantine_reason"] == "JSON_PARSE_ERROR"]
check("p15 is the parse error",
      [json.loads(r["raw_json"]).get("id") for r in parse_error], ["p15"])
no_id = [r for r in q if r["quarantine_reason"] == "MISSING: event_id"]
check("missing-id row found once", len(no_id), 1)
r = no_id[0] if no_id else {}
check("missing-id row has no id in raw_json", "id" in json.loads(r.get("raw_json", "{}")), False)
check("missing-id row PR fields kept",
      (r.get("pr_action"), r.get("pr_number"), r.get("pr_id"), r.get("pr_merged"), sizes(r)),
      ("opened", 42, 9001, False, (3, 10, 2, 1)))
check("missing-id row original text kept",
      (r.get("pr_number_raw"), r.get("pr_id_raw"), r.get("pr_created_at_raw"),
       r.get("pr_closed_at_raw"), r.get("pr_merged_raw"), r.get("pr_additions_raw")),
      ("42", "9001", "2024-01-31T08:00:00Z", None, "false", "10"))
check("quarantine keeps raw_json", all(r["raw_json"] for r in q), True)
check("quarantine keys non-NULL", all(r["quarantine_key"] for r in q), True)

finish("B5 run 1")

## B6. Run 2: rerun the same date

In [0]:
%run ../02_pull_request_events $test_schema="test_pull_request" $source_date="2024-02-01" $run_id="test-run2"

## B7. Run 2 checks: idempotency, plus the final safety check

In [0]:
check_audit({
    "bronze_input_rows": 19, "silver_candidate_rows": 16,
    "silver_rows_inserted": 0, "silver_rows_already_present": 16,
    "quarantine_candidate_rows": 2, "quarantine_rows_inserted": 0,
}, QUALITY_RUN1)
check("silver rows unchanged", spark.table(TEST_SILVER).count(), 16)
check("quarantine rows unchanged", spark.table(TEST_QUARANTINE).count(), 2)

statuses = [r["status"] for r in
            spark.table(TEST_AUDIT).orderBy("run_finished_at").collect()]
check("audit history", statuses, ["SUCCEEDED", "SUCCEEDED"])

for t in PROD_TABLES:
    check(f"prod version unchanged: {t}", table_version(t), prod_versions_before[t])
finish("B7 run 2 + production safety")

## Clean up: drop the test schema

In [0]:
assert TEST_SCHEMA.startswith("test_")
spark.sql(f"DROP SCHEMA IF EXISTS github_lakehouse.{TEST_SCHEMA} CASCADE")